# FDS Pokémon Battles — Final three-submission notebook

Classical machine-learning pipeline using only the official competition files. This notebook clones **lucamartone/fds-pokemon** and regenerates the offense, full baseline and consensus outputs. It does not embed code or previously uploaded test predictions.

**Reproducibility caveat:** The original three selected Kaggle CSVs must be compared with these regenerated outputs before asserting exact equivalence; equal leaderboard scores alone are insufficient.

## 1. Clone our GitHub source at a fixed commit

Enable Internet for Git and package installation. No modeling inputs are downloaded from external sources.

In [ ]:
from pathlib import Path
import subprocess, sys
REPO_URL="https://github.com/lucamartone/fds-pokemon.git"
SOURCE_COMMIT="a10ab4c7913787ee5f931671fa114506e21cd347"
REPO_DIR=Path("/kaggle/working/fds-pokemon")
if not REPO_DIR.exists():
    subprocess.run(["git","clone","--no-checkout",REPO_URL,str(REPO_DIR)],check=True)
else:
    origin=subprocess.check_output(["git","-C",str(REPO_DIR),"remote","get-url","origin"],text=True).strip()
    assert origin==REPO_URL,origin
subprocess.run(["git","-C",str(REPO_DIR),"checkout","--detach",SOURCE_COMMIT],check=True)
assert subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],text=True).strip()==SOURCE_COMMIT
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(REPO_DIR/"requirements.txt")],check=True)
print("Pinned source commit:",SOURCE_COMMIT)

## 2. Locate the official competition data and train

Attach the original `train.jsonl`, `test.jsonl`, `type_chart_gen2.csv` and `sample_submission.csv` from the competition. The shared entrypoint trains the baseline and independent experts once.

In [ ]:
required=("train.jsonl","test.jsonl","type_chart_gen2.csv","sample_submission.csv")
matches=sorted({p.parent for p in Path("/kaggle/input").rglob("train.jsonl") if all((p.parent/f).is_file() for f in required)})
if len(matches)!=1: raise FileNotFoundError(f"Expected one dataset folder, found: {matches}")
DATA_DIR=matches[0]
OUTPUT_DIR=Path("/kaggle/working/submissions")
subprocess.run([sys.executable,str(REPO_DIR/"run.py"),"--data",str(DATA_DIR),"--output-dir",str(OUTPUT_DIR)],cwd=REPO_DIR,check=True)

In [ ]:
import hashlib
import pandas as pd
sample=pd.read_csv(DATA_DIR/"sample_submission.csv")
def inspect(name):
    path=OUTPUT_DIR/name
    df=pd.read_csv(path)
    assert df.shape==sample.shape,name
    assert df.columns.equals(sample.columns),name
    assert df.battle_id.equals(sample.battle_id),name
    assert df.battle_id.is_unique,name
    assert df.player_won.isin([0,1]).all(),name
    print(name, "rows",len(df),"SHA256",hashlib.sha256(path.read_bytes()).hexdigest())
    display(df.head(3))
    return df

## 3. Submission 1 — observed offense

The frozen offense blend combines the original 45/45/10 LightGBM/XGBoost/CatBoost baseline with a compact logistic model, an active-species CatBoost expert and an observed-damaging-moves CatBoost expert. It is the internal `candidate_7001` model; this identifier is not a leaderboard score.

In [ ]:
frame=inspect("submission_1_offense.csv")

## 4. Submission 2 — full baseline

The full boosted ensemble further incorporates observed speed/priority and recovery, status and setup flags. It is internally called `candidate_7010`. All supervised training uses the official labels.

In [ ]:
frame=inspect("submission_2_full.csv")

## 5. Submission 3 — unanimous consensus

The full baseline decision is overridden only within 0.10 of probability 0.5 and only when all three independently fitted roster, hybrid and mirrored-orientation experts agree on the opposite label. No previously scored test predictions are used.

In [ ]:
frame=inspect("submission_3_consensus.csv")

## 6. Reproduction verification

The three outputs are written to `/kaggle/working/submissions/`. This notebook checks their schema, unique IDs, binary labels and SHA-256 hashes. **Exact reproduction remains unconfirmed** until each output is compared with the original CSV submitted to Kaggle, using `scripts/check_submission.py`.

Make the notebook public after competition closure. Submit the public notebook link and the separate methodological report as required by the organizers.